In [1]:
# Run once if needed.
# In a stable environment, prefer doing this in terminal, then restarting Jupyter.

!pip install -U openai anthropic google-genai pandas tqdm python-dotenv

In [2]:
from __future__ import annotations

import os
import re
import json
import time
import hashlib
import itertools
import datetime as dt
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Dict, Any, List

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()

TASK_FAMILY = "slogan"

BASE_DIR = Path(".")
AI_DATA_DIR = Path("ai_data/slogan_generations")
OPENAI_BATCH_DIR = Path("ai_data/slogan_batches/openai")
ANTHROPIC_BATCH_DIR = Path("ai_data/slogan_batches/anthropic")
GEMINI_BATCH_DIR = Path("ai_data/slogan_batches/gemini")
CLEAN_AI_DIR = Path("ai_data/processed")

for p in [AI_DATA_DIR, OPENAI_BATCH_DIR, ANTHROPIC_BATCH_DIR, GEMINI_BATCH_DIR, CLEAN_AI_DIR]:
    p.mkdir(parents=True, exist_ok=True)

In [3]:
from openai import OpenAI
import anthropic
from google import genai
from google.genai import types

openai_client = OpenAI(api_key="")
anthropic_client = anthropic.Anthropic(api_key="")
gemini_client = genai.Client(api_key="")

/Users/raiyanabdulbaten/miniforge3/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/raiyanabdulbaten/miniforge3/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)


In [4]:
SLOGAN_CONDITIONS = [
    {
        "condition_id": "smartphone",
        "condition_label": "Smartphone",
        "product": "smartphone",
        "task_context": "a tech company preparing to launch a new smartphone",
    }
]

slogan_conditions_df = pd.DataFrame(SLOGAN_CONDITIONS)
slogan_conditions_df

,condition_id,condition_label,product,task_context
0,smartphone,Smartphone,smartphone,a tech company preparing to launch a new smart...


In [5]:
def generate_all_personas():
    trait_pairs = [
        ("extroverted", "introverted"),
        ("agreeable", "antagonistic"),
        ("conscientious", "unconscientious"),
        ("neurotic", "emotionally_stable"),
        ("open to experience", "closed to experience"),
    ]
    return list(itertools.product(*trait_pairs))


def persona_tuple_to_id(persona_tuple) -> str:
    return "__".join(
        t.replace(" ", "_").replace("-", "_")
        for t in persona_tuple
    )


def persona_tuple_to_prompt_text(persona_tuple) -> str:
    traits = ", ".join(persona_tuple)
    return (
        "Write as if you are a person with the following personality profile: "
        f"{traits}. Let this personality influence the creative choice, style, tone, "
        "and framing of the slogan, while still following the task exactly."
    )


ALL_PERSONAS = generate_all_personas()

len(ALL_PERSONAS), ALL_PERSONAS[:3]

(32,
 [('extroverted',
   'agreeable',
   'conscientious',
   'neurotic',
   'open to experience'),
  ('extroverted',
   'agreeable',
   'conscientious',
   'neurotic',
   'closed to experience'),
  ('extroverted',
   'agreeable',
   'conscientious',
   'emotionally_stable',
   'open to experience')])

In [6]:
SLOGAN_NEUTRAL_SYSTEM_INSTRUCTIONS = (
    "You are participating in a creativity test. "
    "Follow the task instructions exactly. "
    "Do not explain your reasoning. "
    "Do not include commentary before or after the answer."
)


def build_slogan_system_instructions(
    condition_type: str,
    persona_tuple: Optional[tuple] = None,
) -> str:
    """
    condition_type:
        - neutral
        - personality
    """
    if condition_type == "neutral":
        return SLOGAN_NEUTRAL_SYSTEM_INSTRUCTIONS

    if condition_type == "personality":
        if persona_tuple is None:
            raise ValueError("persona_tuple must be provided for personality condition.")

        return (
            SLOGAN_NEUTRAL_SYSTEM_INSTRUCTIONS
            + "\n\n"
            + persona_tuple_to_prompt_text(persona_tuple)
        )

    raise ValueError(f"Unknown condition_type: {condition_type}")


def build_slogan_user_prompt(product: str = "smartphone") -> str:
    return (
        "You are part of the marketing team at a tech company preparing to launch "
        f"a new {product}.\n\n"
        f"Generate exactly one creative marketing slogan for this brand new {product}.\n\n"
        "Requirements:\n"
        "- The slogan must be novel and appropriate.\n"
        "- The slogan must not exceed 6 words.\n"
        "- You may assume any detail about the smartphone.\n"
        "- Do not list multiple slogans.\n"
        "- Return only the slogan text."
    )


print(build_slogan_user_prompt("smartphone"))

You are part of the marketing team at a tech company preparing to launch a new smartphone.

Generate exactly one creative marketing slogan for this brand new smartphone.

Requirements:
- The slogan must be novel and appropriate.
- The slogan must not exceed 6 words.
- You may assume any detail about the smartphone.
- Do not list multiple slogans.
- Return only the slogan text.


In [7]:
def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def stable_hash(obj: Any) -> str:
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path: str | Path) -> List[Dict[str, Any]]:
    path = Path(path)
    if not path.exists():
        return []
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records


def load_generation_jsonl(path: str | Path) -> pd.DataFrame:
    return pd.DataFrame(read_jsonl(Path(path)))


def successful_request_keys(path: Path) -> set:
    records = read_jsonl(path)
    return {
        r["request_key"]
        for r in records
        if r.get("status") == "success" and r.get("text")
    }


def make_request_key(
    provider: str,
    model: str,
    task_family: str,
    scenario_name: str,
    condition_id: str,
    condition_type: str,
    temperature: float,
    run_idx: int,
    persona_id: Optional[str],
    system_instructions: str,
    user_prompt: str,
) -> str:
    return stable_hash(
        {
            "provider": provider,
            "model": model,
            "task_family": task_family,
            "scenario_name": scenario_name,
            "condition_id": condition_id,
            "condition_type": condition_type,
            "temperature": float(temperature),
            "run_idx": int(run_idx),
            "persona_id": persona_id,
            "system_instructions": system_instructions,
            "user_prompt": user_prompt,
        }
    )

In [8]:
@dataclass
class GenerationScenario:
    scenario_name: str
    condition_type: str
    temperatures: List[float]
    n_per_condition: int
    condition_ids: List[str]
    include_personas: bool = False
    max_output_tokens: int = 40
    description: str = ""


@dataclass
class ProviderModelConfig:
    provider: str
    model: str
    batch_dir: Path
    temperature_grid: List[float]
    neutral_temperature_robustness_grid: List[float]
    personality_temperature_grid: List[float]
    max_output_tokens: int = 40


MODEL_CONFIGS = {
    "openai_gpt54": ProviderModelConfig(
        provider="openai",
        model="gpt-5.4",
        batch_dir=OPENAI_BATCH_DIR,
        temperature_grid=[0.7, 1.0, 1.3],
        neutral_temperature_robustness_grid=[0.7, 1.3],
        personality_temperature_grid=[0.7, 1.0, 1.3],
        max_output_tokens=40,
    ),
    "claude_sonnet45": ProviderModelConfig(
        provider="anthropic",
        model="claude-sonnet-4-5",
        batch_dir=ANTHROPIC_BATCH_DIR,
        temperature_grid=[0.3, 0.7, 1.0],
        neutral_temperature_robustness_grid=[0.3, 0.7],
        personality_temperature_grid=[0.3, 0.7, 1.0],
        max_output_tokens=40,
    ),
    "gemini_flash25": ProviderModelConfig(
        provider="gemini",
        model="gemini-2.5-flash",
        batch_dir=GEMINI_BATCH_DIR,
        temperature_grid=[0.7, 1.0, 1.3],
        neutral_temperature_robustness_grid=[0.7, 1.3],
        personality_temperature_grid=[0.7, 1.0, 1.3],
        max_output_tokens=40,
    ),
}


def build_scenarios_for_provider(config: ProviderModelConfig) -> Dict[str, GenerationScenario]:
    condition_ids = slogan_conditions_df["condition_id"].tolist()

    return {
        "neutral_main_t1": GenerationScenario(
            scenario_name="neutral_main_t1",
            condition_type="neutral",
            temperatures=[1.0],
            n_per_condition=50,
            condition_ids=condition_ids,
            include_personas=False,
            max_output_tokens=config.max_output_tokens,
            description="Primary slogan benchmark: neutral prompt, temperature 1.0, 50 generations.",
        ),
        "neutral_temperature_robustness": GenerationScenario(
            scenario_name="neutral_temperature_robustness",
            condition_type="neutral",
            temperatures=config.neutral_temperature_robustness_grid,
            n_per_condition=10,
            condition_ids=condition_ids,
            include_personas=False,
            max_output_tokens=config.max_output_tokens,
            description="Slogan temperature robustness: neutral prompt at non-default temperatures.",
        ),
        "personality_grid": GenerationScenario(
            scenario_name="personality_grid",
            condition_type="personality",
            temperatures=config.personality_temperature_grid,
            n_per_condition=10,
            condition_ids=condition_ids,
            include_personas=True,
            max_output_tokens=config.max_output_tokens,
            description="Slogan personality robustness: 32 Big-Five-style profiles crossed with temperature.",
        ),
    }


def expected_request_count(config: ProviderModelConfig) -> pd.DataFrame:
    scenarios = build_scenarios_for_provider(config)
    rows = []
    for name, s in scenarios.items():
        n_personas = len(ALL_PERSONAS) if s.include_personas else 1
        rows.append({
            "scenario_name": name,
            "n_conditions": len(s.condition_ids),
            "n_temperatures": len(s.temperatures),
            "n_personas": n_personas,
            "n_per_condition": s.n_per_condition,
            "n_requests": len(s.condition_ids) * len(s.temperatures) * n_personas * s.n_per_condition,
        })
    return pd.DataFrame(rows)


for name, cfg in MODEL_CONFIGS.items():
    print("=" * 80)
    print(name)
    display(expected_request_count(cfg))

openai_gpt54


,scenario_name,n_conditions,n_temperatures,n_personas,n_per_condition,n_requests
0,neutral_main_t1,1,1,1,50,50
1,neutral_temperature_robustness,1,2,1,10,20
2,personality_grid,1,3,32,10,960


claude_sonnet45


,scenario_name,n_conditions,n_temperatures,n_personas,n_per_condition,n_requests
0,neutral_main_t1,1,1,1,50,50
1,neutral_temperature_robustness,1,2,1,10,20
2,personality_grid,1,3,32,10,960


gemini_flash25


,scenario_name,n_conditions,n_temperatures,n_personas,n_per_condition,n_requests
0,neutral_main_t1,1,1,1,50,50
1,neutral_temperature_robustness,1,2,1,10,20
2,personality_grid,1,3,32,10,960


In [9]:
def build_slogan_generation_plan(
    scenario: GenerationScenario,
    conditions_df: pd.DataFrame,
    provider_name: str,
    model_name: str,
) -> pd.DataFrame:
    rows = []

    conditions_use = (
        conditions_df[conditions_df["condition_id"].isin(scenario.condition_ids)]
        .copy()
        .sort_values("condition_id")
    )

    persona_tuples = ALL_PERSONAS if scenario.include_personas else [None]

    for _, condition_row in conditions_use.iterrows():
        condition_id = condition_row["condition_id"]
        condition_label = condition_row["condition_label"]
        product = condition_row["product"]
        task_context = condition_row["task_context"]
        user_prompt = build_slogan_user_prompt(product=product)

        for temperature in scenario.temperatures:
            for persona_tuple in persona_tuples:
                if persona_tuple is None:
                    persona_id = None
                    persona_traits = None
                else:
                    persona_id = persona_tuple_to_id(persona_tuple)
                    persona_traits = list(persona_tuple)

                system_instructions = build_slogan_system_instructions(
                    condition_type=scenario.condition_type,
                    persona_tuple=persona_tuple,
                )

                for run_idx in range(scenario.n_per_condition):
                    request_key = make_request_key(
                        provider=provider_name,
                        model=model_name,
                        task_family=TASK_FAMILY,
                        scenario_name=scenario.scenario_name,
                        condition_id=condition_id,
                        condition_type=scenario.condition_type,
                        temperature=float(temperature),
                        run_idx=int(run_idx),
                        persona_id=persona_id,
                        system_instructions=system_instructions,
                        user_prompt=user_prompt,
                    )

                    rows.append({
                        "request_key": request_key,
                        "task_family": TASK_FAMILY,
                        "scenario_name": scenario.scenario_name,
                        "provider": provider_name,
                        "model": model_name,
                        "condition_type": scenario.condition_type,
                        "condition_id": condition_id,
                        "condition_label": condition_label,
                        "product": product,
                        "task_context": task_context,
                        "temperature": float(temperature),
                        "run_idx": int(run_idx),
                        "persona_id": persona_id,
                        "persona_traits": persona_traits,
                        "system_instructions": system_instructions,
                        "user_prompt": user_prompt,
                        "max_output_tokens": int(scenario.max_output_tokens),
                    })

    return pd.DataFrame(rows)


cfg = MODEL_CONFIGS["openai_gpt54"]
scenarios = build_scenarios_for_provider(cfg)

test_plan = build_slogan_generation_plan(
    scenario=scenarios["neutral_main_t1"],
    conditions_df=slogan_conditions_df,
    provider_name=cfg.provider,
    model_name=cfg.model,
)

test_plan.shape, test_plan.head()

((50, 17),
                                          request_key task_family  \
 0  8c4f2c6fbabf8d378ca3f6524638ea1b67ef20d08819e2...      slogan   
 1  b1a6e689c259ff82f88588ce77b5d0e9aa7d0b2423bec5...      slogan   
 2  a526b642313a496a85619592abdaf95418524ba6975781...      slogan   
 3  92f0d36c64ed3f638fff274725af04e443fce7aee78ba5...      slogan   
 4  b70f4055e3e66d9a223a0ffdbe383ac5c489bdf19e9618...      slogan   
 
      scenario_name provider    model condition_type condition_id  \
 0  neutral_main_t1   openai  gpt-5.4        neutral   smartphone   
 1  neutral_main_t1   openai  gpt-5.4        neutral   smartphone   
 2  neutral_main_t1   openai  gpt-5.4        neutral   smartphone   
 3  neutral_main_t1   openai  gpt-5.4        neutral   smartphone   
 4  neutral_main_t1   openai  gpt-5.4        neutral   smartphone   
 
   condition_label     product  \
 0      Smartphone  smartphone   
 1      Smartphone  smartphone   
 2      Smartphone  smartphone   
 3      Smartphone  s

In [10]:
def load_all_generation_files(
    provider_name: str,
    model_name: str,
    output_dir: Path = AI_DATA_DIR,
) -> pd.DataFrame:
    paths = sorted(output_dir.glob(f"{provider_name}__{model_name}__*.jsonl"))
    paths = [p for p in paths if "__plan" not in p.name]

    dfs = []
    for p in paths:
        df = load_generation_jsonl(p)
        if len(df) > 0:
            df["source_file"] = str(p)
            dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)


def deduplicate_generation_records(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    df = df.copy()
    status_priority = {"success": 3, "empty_text": 2, "error": 1}
    df["_status_priority"] = df["status"].map(status_priority).fillna(0)
    df["_original_order"] = range(len(df))

    df = (
        df.sort_values(["request_key", "_status_priority", "_original_order"])
        .drop_duplicates("request_key", keep="last")
        .drop(columns=["_status_priority", "_original_order"])
        .reset_index(drop=True)
    )

    return df


def clean_slogan_for_diagnostics(x):
    if not isinstance(x, str):
        return ""
    x = x.strip()
    x = re.sub(r"^['\"“”‘’]+|['\"“”‘’]+$", "", x).strip()
    x = re.sub(r"\s+", " ", x)
    return x


def add_slogan_validity_diagnostics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    def word_count(x):
        x = clean_slogan_for_diagnostics(x)
        if not x:
            return 0
        return len(re.findall(r"\b[\w']+\b", x))

    def line_count(x):
        if not isinstance(x, str):
            return 0
        return len([ln for ln in x.splitlines() if ln.strip()])

    def looks_like_list(x):
        if not isinstance(x, str):
            return False
        lines = [ln.strip() for ln in x.splitlines() if ln.strip()]
        if len(lines) > 1:
            return True
        return bool(re.search(r"(^|\n)\s*(\d+[\.\)]|[-*•])\s+", x))

    def has_commentary(x):
        if not isinstance(x, str):
            return False
        lower = x.lower().strip()
        commentary_patterns = [
            r"^here('s| is)\b",
            r"^slogan:",
            r"^sure[,! ]",
            r"^my slogan",
            r"^option",
        ]
        return any(re.search(p, lower) for p in commentary_patterns)

    def likely_multiple_slogans(x):
        if not isinstance(x, str):
            return False
        if looks_like_list(x):
            return True
        # Multiple quoted slogans or slash-separated alternatives can signal multiple slogans.
        if len(re.findall(r'["“][^"”]+["”]', x)) > 1:
            return True
        if ";" in x:
            return True
        return False

    df["slogan_clean"] = df["text"].map(clean_slogan_for_diagnostics)
    df["output_word_count"] = df["text"].map(word_count)
    df["output_line_count"] = df["text"].map(line_count)
    df["looks_like_list"] = df["text"].map(looks_like_list)
    df["likely_multiple_slogans"] = df["text"].map(likely_multiple_slogans)
    df["has_commentary"] = df["text"].map(has_commentary)

    df["valid_one_slogan_heuristic"] = (
        (df["status"] == "success")
        & (df["output_word_count"].between(1, 6))
        & (~df["likely_multiple_slogans"])
        & (~df["has_commentary"])
    )

    return df


def save_provider_outputs(provider: str, model: str, df: pd.DataFrame):
    audit_pkl = CLEAN_AI_DIR / f"{provider}__{model}__slogan_generations_all_records_with_errors.pkl"
    audit_csv = CLEAN_AI_DIR / f"{provider}__{model}__slogan_generations_all_records_with_errors.csv"
    success_pkl = CLEAN_AI_DIR / f"{provider}__{model}__slogan_generations_success_only.pkl"
    success_csv = CLEAN_AI_DIR / f"{provider}__{model}__slogan_generations_success_only.csv"

    df = add_slogan_validity_diagnostics(df)
    success_df = df.query("status == 'success'").copy()

    df.to_pickle(audit_pkl)
    df.to_csv(audit_csv, index=False)
    success_df.to_pickle(success_pkl)
    success_df.to_csv(success_csv, index=False)

    print("Saved:")
    print(audit_pkl)
    print(audit_csv)
    print(success_pkl)
    print(success_csv)

    return {
        "audit_pkl": audit_pkl,
        "audit_csv": audit_csv,
        "success_pkl": success_pkl,
        "success_csv": success_csv,
    }


def summarize_provider_outputs(provider: str, model: str) -> pd.DataFrame:
    df = load_all_generation_files(provider, model, AI_DATA_DIR)
    df = deduplicate_generation_records(df)
    if df.empty:
        return pd.DataFrame()

    df = add_slogan_validity_diagnostics(df)

    summary = (
        df.groupby(["scenario_name", "condition_id", "temperature", "status"], dropna=False)
        .size()
        .reset_index(name="n")
        .sort_values(["scenario_name", "condition_id", "temperature", "status"])
    )

    display(summary)
    return df

# OpenAI

In [11]:
def make_openai_batch_input_jsonl(
    scenario_name: str,
    config: ProviderModelConfig,
    conditions_df: pd.DataFrame,
    output_dir: Path = OPENAI_BATCH_DIR,
    exclude_existing_successes: bool = True,
) -> tuple[Path, Path, pd.DataFrame]:
    scenarios = build_scenarios_for_provider(config)
    scenario = scenarios[scenario_name]

    plan_df = build_slogan_generation_plan(
        scenario=scenario,
        conditions_df=conditions_df,
        provider_name=config.provider,
        model_name=config.model,
    )

    existing_output_path = AI_DATA_DIR / f"{config.provider}__{config.model}__{scenario_name}.jsonl"
    if exclude_existing_successes and existing_output_path.exists():
        done_keys = successful_request_keys(existing_output_path)
        plan_df = plan_df[~plan_df["request_key"].isin(done_keys)].copy()

    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"{config.provider}__{config.model}__{TASK_FAMILY}__{scenario_name}__{timestamp}"

    batch_jsonl_path = output_dir / f"{stem}__batch_input.jsonl"
    plan_csv_path = output_dir / f"{stem}__batch_plan.csv"

    plan_df.to_csv(plan_csv_path, index=False)

    with open(batch_jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            body = {
                "model": config.model,
                "instructions": row["system_instructions"],
                "input": row["user_prompt"],
                "temperature": float(row["temperature"]),
                "max_output_tokens": int(row["max_output_tokens"]),
            }

            request = {
                "custom_id": row["request_key"],
                "method": "POST",
                "url": "/v1/responses",
                "body": body,
            }

            f.write(json.dumps(request, ensure_ascii=False) + "\n")

    print(f"OpenAI scenario: {scenario_name}")
    print(f"Requests written: {len(plan_df):,}")
    print(f"Batch JSONL: {batch_jsonl_path}")
    print(f"Plan CSV:    {plan_csv_path}")

    return batch_jsonl_path, plan_csv_path, plan_df


def submit_openai_batch(batch_jsonl_path: Path, scenario_name: str, model_name: str) -> dict:
    batch_input_file = openai_client.files.create(
        file=open(batch_jsonl_path, "rb"),
        purpose="batch",
    )

    batch = openai_client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={
            "task_family": TASK_FAMILY,
            "scenario_name": scenario_name,
            "model": model_name,
            "local_input_file": str(batch_jsonl_path),
        },
    )

    batch_info = {
        "batch_id": batch.id,
        "input_file_id": batch_input_file.id,
        "scenario_name": scenario_name,
        "model": model_name,
        "submitted_at_utc": now_iso(),
        "local_input_file": str(batch_jsonl_path),
    }

    manifest_path = batch_jsonl_path.with_suffix(".batch_manifest.json")
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(batch_info, f, indent=2)

    print("Submitted OpenAI batch:")
    print(json.dumps(batch_info, indent=2))

    return batch_info


def check_openai_batch(batch_id: str):
    batch = openai_client.batches.retrieve(batch_id)
    print(batch)
    return batch


def download_openai_batch_results(batch_id: str, output_dir: Path = OPENAI_BATCH_DIR):
    batch = openai_client.batches.retrieve(batch_id)

    if batch.status != "completed":
        print(f"Batch is not completed yet. Current status: {batch.status}")
        return None, None

    output_path = output_dir / f"{batch_id}__output.jsonl"
    error_path = output_dir / f"{batch_id}__errors.jsonl"

    if batch.output_file_id:
        file_response = openai_client.files.content(batch.output_file_id)
        output_path.write_text(file_response.text, encoding="utf-8")
        print(f"Downloaded output: {output_path}")

    if batch.error_file_id:
        error_response = openai_client.files.content(batch.error_file_id)
        error_path.write_text(error_response.text, encoding="utf-8")
        print(f"Downloaded errors: {error_path}")
    else:
        error_path = None

    return output_path, error_path


def extract_text_from_responses_api_body(body: dict) -> str:
    if not isinstance(body, dict):
        return ""

    if body.get("output_text"):
        return str(body["output_text"]).strip()

    texts = []
    for item in body.get("output", []) or []:
        for content in item.get("content", []) or []:
            if content.get("type") in {"output_text", "text"} and "text" in content:
                texts.append(content["text"])

    return "\n".join(texts).strip()


def parse_openai_batch_output_to_standard_jsonl(
    batch_output_path: Path,
    plan_csv_path: Path,
    scenario_name: str,
    config: ProviderModelConfig,
    standard_output_dir: Path = AI_DATA_DIR,
) -> Path:
    plan_df = pd.read_csv(plan_csv_path)
    plan_by_key = {
        row["request_key"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    standard_output_path = standard_output_dir / f"{config.provider}__{config.model}__{scenario_name}.jsonl"
    batch_records = read_jsonl(batch_output_path)

    n_success = 0
    n_error = 0

    for rec in batch_records:
        request_key = rec.get("custom_id")
        plan_row = plan_by_key.get(request_key, {})
        response = rec.get("response")
        error = rec.get("error")

        if response and response.get("body"):
            body = response["body"]
            text = extract_text_from_responses_api_body(body)
            usage = body.get("usage")

            record = {
                **plan_row,
                "created_at_utc": now_iso(),
                "status": "success" if text else "empty_text",
                "text": text if text else None,
                "provider_response_id": body.get("id"),
                "usage": usage,
                "raw_response": None,
                "error": None if text else "No text extracted from response body.",
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
            }

            n_success += int(bool(text))
            n_error += int(not bool(text))
        else:
            record = {
                **plan_row,
                "created_at_utc": now_iso(),
                "status": "error",
                "text": None,
                "provider_response_id": None,
                "usage": None,
                "raw_response": None,
                "error": error,
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
            }
            n_error += 1

        append_jsonl(standard_output_path, record)

    print(f"Parsed OpenAI batch output to: {standard_output_path}")
    print(f"Success-ish text records: {n_success:,}")
    print(f"Errors/empty:             {n_error:,}")

    return standard_output_path


def submit_openai_scenario(scenario_name: str, exclude_existing_successes: bool = True) -> dict:
    cfg = MODEL_CONFIGS["openai_gpt54"]

    batch_jsonl_path, plan_csv_path, plan_df = make_openai_batch_input_jsonl(
        scenario_name=scenario_name,
        config=cfg,
        conditions_df=slogan_conditions_df,
        output_dir=OPENAI_BATCH_DIR,
        exclude_existing_successes=exclude_existing_successes,
    )

    if len(plan_df) == 0:
        print(f"No remaining requests for {scenario_name}.")
        return {
            "scenario_name": scenario_name,
            "model": cfg.model,
            "provider": cfg.provider,
            "n_requests_submitted": 0,
            "plan_csv_path": str(plan_csv_path),
            "batch_jsonl_path": str(batch_jsonl_path),
            "batch_id": None,
        }

    batch_info = submit_openai_batch(
        batch_jsonl_path=batch_jsonl_path,
        scenario_name=scenario_name,
        model_name=cfg.model,
    )

    batch_info["provider"] = cfg.provider
    batch_info["plan_csv_path"] = str(plan_csv_path)
    batch_info["batch_jsonl_path"] = str(batch_jsonl_path)
    batch_info["n_requests_submitted"] = len(plan_df)

    return batch_info


def parse_completed_openai_batch(batch_info: dict):
    cfg = MODEL_CONFIGS["openai_gpt54"]
    batch_id = batch_info["batch_id"]

    output_path, error_path = download_openai_batch_results(batch_id, OPENAI_BATCH_DIR)
    if output_path is None:
        return None

    standard_path = parse_openai_batch_output_to_standard_jsonl(
        batch_output_path=output_path,
        plan_csv_path=Path(batch_info["plan_csv_path"]),
        scenario_name=batch_info["scenario_name"],
        config=cfg,
    )

    return standard_path

In [12]:
openai_main_batch = submit_openai_scenario("neutral_main_t1")
openai_main_batch

OpenAI scenario: neutral_main_t1
Requests written: 50
Batch JSONL: ai_data/slogan_batches/openai/openai__gpt-5.4__slogan__neutral_main_t1__20260429_162109__batch_input.jsonl
Plan CSV:    ai_data/slogan_batches/openai/openai__gpt-5.4__slogan__neutral_main_t1__20260429_162109__batch_plan.csv
Submitted OpenAI batch:
{
  "batch_id": "batch_69f2683735748190abc6900d88967449",
  "input_file_id": "file-5Csh4FGTf9TnmaBq4UuTDQ",
  "scenario_name": "neutral_main_t1",
  "model": "gpt-5.4",
  "submitted_at_utc": "2026-04-29T20:21:11.969986+00:00",
  "local_input_file": "ai_data/slogan_batches/openai/openai__gpt-5.4__slogan__neutral_main_t1__20260429_162109__batch_input.jsonl"
}


{'batch_id': 'batch_69f2683735748190abc6900d88967449',
 'input_file_id': 'file-5Csh4FGTf9TnmaBq4UuTDQ',
 'scenario_name': 'neutral_main_t1',
 'model': 'gpt-5.4',
 'submitted_at_utc': '2026-04-29T20:21:11.969986+00:00',
 'local_input_file': 'ai_data/slogan_batches/openai/openai__gpt-5.4__slogan__neutral_main_t1__20260429_162109__batch_input.jsonl',
 'provider': 'openai',
 'plan_csv_path': 'ai_data/slogan_batches/openai/openai__gpt-5.4__slogan__neutral_main_t1__20260429_162109__batch_plan.csv',
 'batch_jsonl_path': 'ai_data/slogan_batches/openai/openai__gpt-5.4__slogan__neutral_main_t1__20260429_162109__batch_input.jsonl',
 'n_requests_submitted': 50}

In [19]:
check_openai_batch(openai_main_batch["batch_id"])

Batch(id='batch_69f2683735748190abc6900d88967449', completion_window='24h', created_at=1777494071, endpoint='/v1/responses', input_file_id='file-5Csh4FGTf9TnmaBq4UuTDQ', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1777495062, error_file_id=None, errors=None, expired_at=None, expires_at=1777580471, failed_at=None, finalizing_at=1777495054, in_progress_at=1777494133, metadata={'task_family': 'slogan', 'scenario_name': 'neutral_main_t1', 'model': 'gpt-5.4', 'local_input_file': 'ai_data/slogan_batches/openai/openai__gpt-5.4__slogan__neutral_main_t1__20260429_162109__batch_input.jsonl'}, model='gpt-5.4-2026-03-05', output_file_id='file-VhkTH87mwACNPpCoyBzU1Q', request_counts=BatchRequestCounts(completed=50, failed=0, total=50), usage=BatchUsage(input_tokens=5750, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=538, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=6288))


Batch(id='batch_69f2683735748190abc6900d88967449', completion_window='24h', created_at=1777494071, endpoint='/v1/responses', input_file_id='file-5Csh4FGTf9TnmaBq4UuTDQ', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1777495062, error_file_id=None, errors=None, expired_at=None, expires_at=1777580471, failed_at=None, finalizing_at=1777495054, in_progress_at=1777494133, metadata={'task_family': 'slogan', 'scenario_name': 'neutral_main_t1', 'model': 'gpt-5.4', 'local_input_file': 'ai_data/slogan_batches/openai/openai__gpt-5.4__slogan__neutral_main_t1__20260429_162109__batch_input.jsonl'}, model='gpt-5.4-2026-03-05', output_file_id='file-VhkTH87mwACNPpCoyBzU1Q', request_counts=BatchRequestCounts(completed=50, failed=0, total=50), usage=BatchUsage(input_tokens=5750, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=538, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=6288))

In [20]:
openai_main_standard_path = parse_completed_openai_batch(openai_main_batch)

openai_main_df = load_generation_jsonl(openai_main_standard_path)
print(openai_main_df.shape)
print(openai_main_df["status"].value_counts(dropna=False))
display(openai_main_df[["condition_id", "temperature", "run_idx", "status", "text"]].head(10))

Downloaded output: ai_data/slogan_batches/openai/batch_69f2683735748190abc6900d88967449__output.jsonl
Parsed OpenAI batch output to: ai_data/slogan_generations/openai__gpt-5.4__neutral_main_t1.jsonl
Success-ish text records: 50
Errors/empty:             0
(50, 26)
status
success    50
Name: count, dtype: int64


,condition_id,temperature,run_idx,status,text
0,smartphone,1.0,0,success,"Tomorrow, In Your Palm"
1,smartphone,1.0,1,success,"Tomorrow, perfectly in your palm."
2,smartphone,1.0,2,success,"Tomorrow, perfectly in your palm."
3,smartphone,1.0,3,success,"Tomorrow, perfectly in your palm."
4,smartphone,1.0,4,success,"Power Tomorrow, In Your Palm"
5,smartphone,1.0,5,success,"Tomorrow, Perfected in Your Palm"
6,smartphone,1.0,6,success,"Tomorrow, perfectly in your palm."
7,smartphone,1.0,7,success,"Tomorrow, perfectly in your palm."
8,smartphone,1.0,8,success,"Tomorrow, perfectly in your palm."
9,smartphone,1.0,9,success,"Tomorrow, perfectly in your palm."


In [21]:
openai_main_df = add_slogan_validity_diagnostics(openai_main_df)

openai_main_df.groupby("condition_id").agg(
    n=("text", "size"),
    n_success=("status", lambda x: (x == "success").sum()),
    mean_words=("output_word_count", "mean"),
    median_words=("output_word_count", "median"),
    validity_rate=("valid_one_slogan_heuristic", "mean"),
)

,n,n_success,mean_words,median_words,validity_rate
condition_id,,,,,
smartphone,50,50,4.86,5.0,1.0


In [22]:
openai_temp_batch = submit_openai_scenario("neutral_temperature_robustness")
openai_personality_batch = submit_openai_scenario("personality_grid")

openai_batches = pd.DataFrame([openai_main_batch, openai_temp_batch, openai_personality_batch])
openai_manifest_path = OPENAI_BATCH_DIR / "openai__gpt-5.4__slogan_batch_manifest.csv"
openai_batches.to_csv(openai_manifest_path, index=False)

openai_batches

OpenAI scenario: neutral_temperature_robustness
Requests written: 20
Batch JSONL: ai_data/slogan_batches/openai/openai__gpt-5.4__slogan__neutral_temperature_robustness__20260429_164008__batch_input.jsonl
Plan CSV:    ai_data/slogan_batches/openai/openai__gpt-5.4__slogan__neutral_temperature_robustness__20260429_164008__batch_plan.csv
Submitted OpenAI batch:
{
  "batch_id": "batch_69f26ca988948190bd59a20b78f6f3a4",
  "input_file_id": "file-JQS9QePn7VmcwE3QdyTWAQ",
  "scenario_name": "neutral_temperature_robustness",
  "model": "gpt-5.4",
  "submitted_at_utc": "2026-04-29T20:40:09.637028+00:00",
  "local_input_file": "ai_data/slogan_batches/openai/openai__gpt-5.4__slogan__neutral_temperature_robustness__20260429_164008__batch_input.jsonl"
}
OpenAI scenario: personality_grid
Requests written: 960
Batch JSONL: ai_data/slogan_batches/openai/openai__gpt-5.4__slogan__personality_grid__20260429_164009__batch_input.jsonl
Plan CSV:    ai_data/slogan_batches/openai/openai__gpt-5.4__slogan__person

,batch_id,input_file_id,scenario_name,model,submitted_at_utc,local_input_file,provider,plan_csv_path,batch_jsonl_path,n_requests_submitted
0,batch_69f2683735748190abc6900d88967449,file-5Csh4FGTf9TnmaBq4UuTDQ,neutral_main_t1,gpt-5.4,2026-04-29T20:21:11.969986+00:00,ai_data/slogan_batches/openai/openai__gpt-5.4_...,openai,ai_data/slogan_batches/openai/openai__gpt-5.4_...,ai_data/slogan_batches/openai/openai__gpt-5.4_...,50
1,batch_69f26ca988948190bd59a20b78f6f3a4,file-JQS9QePn7VmcwE3QdyTWAQ,neutral_temperature_robustness,gpt-5.4,2026-04-29T20:40:09.637028+00:00,ai_data/slogan_batches/openai/openai__gpt-5.4_...,openai,ai_data/slogan_batches/openai/openai__gpt-5.4_...,ai_data/slogan_batches/openai/openai__gpt-5.4_...,20
2,batch_69f26caa3a1081909c3ffdcedbbe8149,file-82f6zGee6s86yrUnjSAtMs,personality_grid,gpt-5.4,2026-04-29T20:40:10.388633+00:00,ai_data/slogan_batches/openai/openai__gpt-5.4_...,openai,ai_data/slogan_batches/openai/openai__gpt-5.4_...,ai_data/slogan_batches/openai/openai__gpt-5.4_...,960


In [30]:
for _, row in openai_batches.dropna(subset=["batch_id"]).iterrows():
    print("=" * 100)
    print(row["scenario_name"])
    check_openai_batch(row["batch_id"])

neutral_main_t1
Batch(id='batch_69f2683735748190abc6900d88967449', completion_window='24h', created_at=1777494071, endpoint='/v1/responses', input_file_id='file-5Csh4FGTf9TnmaBq4UuTDQ', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1777495062, error_file_id=None, errors=None, expired_at=None, expires_at=1777580471, failed_at=None, finalizing_at=1777495054, in_progress_at=1777494133, metadata={'task_family': 'slogan', 'scenario_name': 'neutral_main_t1', 'model': 'gpt-5.4', 'local_input_file': 'ai_data/slogan_batches/openai/openai__gpt-5.4__slogan__neutral_main_t1__20260429_162109__batch_input.jsonl'}, model='gpt-5.4-2026-03-05', output_file_id='file-VhkTH87mwACNPpCoyBzU1Q', request_counts=BatchRequestCounts(completed=50, failed=0, total=50), usage=BatchUsage(input_tokens=5750, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=538, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=6288))
neutral_tem

In [31]:
for batch_info in [openai_temp_batch, openai_personality_batch]:
    parse_completed_openai_batch(batch_info)

Downloaded output: ai_data/slogan_batches/openai/batch_69f26ca988948190bd59a20b78f6f3a4__output.jsonl
Parsed OpenAI batch output to: ai_data/slogan_generations/openai__gpt-5.4__neutral_temperature_robustness.jsonl
Success-ish text records: 20
Errors/empty:             0
Downloaded output: ai_data/slogan_batches/openai/batch_69f26caa3a1081909c3ffdcedbbe8149__output.jsonl
Parsed OpenAI batch output to: ai_data/slogan_generations/openai__gpt-5.4__personality_grid.jsonl
Success-ish text records: 960
Errors/empty:             0


In [32]:
openai_all_df = load_all_generation_files("openai", "gpt-5.4", AI_DATA_DIR)
openai_all_df = deduplicate_generation_records(openai_all_df)
openai_all_df = add_slogan_validity_diagnostics(openai_all_df)

print(openai_all_df["scenario_name"].value_counts(dropna=False))
print(openai_all_df["status"].value_counts(dropna=False))

save_provider_outputs("openai", "gpt-5.4", openai_all_df)

scenario_name
personality_grid                  960
neutral_main_t1                    50
neutral_temperature_robustness     20
Name: count, dtype: int64
status
success    1030
Name: count, dtype: int64
Saved:
ai_data/processed/openai__gpt-5.4__slogan_generations_all_records_with_errors.pkl
ai_data/processed/openai__gpt-5.4__slogan_generations_all_records_with_errors.csv
ai_data/processed/openai__gpt-5.4__slogan_generations_success_only.pkl
ai_data/processed/openai__gpt-5.4__slogan_generations_success_only.csv


{'audit_pkl': PosixPath('ai_data/processed/openai__gpt-5.4__slogan_generations_all_records_with_errors.pkl'),
 'audit_csv': PosixPath('ai_data/processed/openai__gpt-5.4__slogan_generations_all_records_with_errors.csv'),
 'success_pkl': PosixPath('ai_data/processed/openai__gpt-5.4__slogan_generations_success_only.pkl'),
 'success_csv': PosixPath('ai_data/processed/openai__gpt-5.4__slogan_generations_success_only.csv')}

# Claude

In [33]:
def make_anthropic_custom_id(request_key: str) -> str:
    return "r_" + request_key[:48]


def build_anthropic_generation_plan(
    scenario_name: str,
    config: ProviderModelConfig,
    conditions_df: pd.DataFrame,
) -> pd.DataFrame:
    scenarios = build_scenarios_for_provider(config)
    scenario = scenarios[scenario_name]

    plan_df = build_slogan_generation_plan(
        scenario=scenario,
        conditions_df=conditions_df,
        provider_name=config.provider,
        model_name=config.model,
    ).copy()

    plan_df["anthropic_custom_id"] = plan_df["request_key"].map(make_anthropic_custom_id)

    if plan_df["anthropic_custom_id"].duplicated().any():
        raise ValueError("Duplicate anthropic_custom_id found. Increase hash length.")

    return plan_df


def make_anthropic_batch_requests(plan_df: pd.DataFrame) -> list:
    requests = []

    for _, row in plan_df.iterrows():
        params = {
            "model": row["model"],
            "max_tokens": int(row["max_output_tokens"]),
            "temperature": float(row["temperature"]),
            "system": row["system_instructions"],
            "messages": [
                {
                    "role": "user",
                    "content": row["user_prompt"],
                }
            ],
        }

        requests.append({
            "custom_id": row["anthropic_custom_id"],
            "params": params,
        })

    return requests


def submit_anthropic_batch(
    scenario_name: str,
    config: ProviderModelConfig,
    conditions_df: pd.DataFrame,
    output_dir: Path = ANTHROPIC_BATCH_DIR,
    exclude_existing_successes: bool = True,
) -> dict:
    standard_output_path = AI_DATA_DIR / f"{config.provider}__{config.model}__{scenario_name}.jsonl"

    plan_df = build_anthropic_generation_plan(
        scenario_name=scenario_name,
        config=config,
        conditions_df=conditions_df,
    )

    if exclude_existing_successes and standard_output_path.exists():
        done_keys = successful_request_keys(standard_output_path)
        plan_df = plan_df[~plan_df["request_key"].isin(done_keys)].copy()

    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"{config.provider}__{config.model}__{TASK_FAMILY}__{scenario_name}__{timestamp}"

    plan_csv_path = output_dir / f"{stem}__batch_plan.csv"
    request_json_path = output_dir / f"{stem}__batch_requests.json"

    plan_df.to_csv(plan_csv_path, index=False)

    requests = make_anthropic_batch_requests(plan_df)

    with open(request_json_path, "w", encoding="utf-8") as f:
        json.dump(requests, f, indent=2, ensure_ascii=False)

    print(f"Anthropic scenario: {scenario_name}")
    print(f"Requests to submit: {len(requests):,}")
    print(f"Plan CSV: {plan_csv_path}")
    print(f"Request JSON: {request_json_path}")

    if len(requests) == 0:
        return {
            "batch_id": None,
            "scenario_name": scenario_name,
            "provider": config.provider,
            "model": config.model,
            "n_requests_submitted": 0,
            "plan_csv_path": str(plan_csv_path),
            "request_json_path": str(request_json_path),
        }

    batch = anthropic_client.messages.batches.create(requests=requests)

    batch_info = {
        "batch_id": batch.id,
        "scenario_name": scenario_name,
        "provider": config.provider,
        "model": config.model,
        "submitted_at_utc": now_iso(),
        "n_requests_submitted": len(requests),
        "plan_csv_path": str(plan_csv_path),
        "request_json_path": str(request_json_path),
        "processing_status": getattr(batch, "processing_status", None),
    }

    manifest_path = output_dir / f"{stem}__batch_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(batch_info, f, indent=2)

    print("Submitted Anthropic batch:")
    print(json.dumps(batch_info, indent=2))

    return batch_info


def check_anthropic_batch(batch_id: str):
    batch = anthropic_client.messages.batches.retrieve(batch_id)
    print(batch)
    return batch


def check_anthropic_batches(batch_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in batch_df.dropna(subset=["batch_id"]).iterrows():
        batch = anthropic_client.messages.batches.retrieve(row["batch_id"])
        request_counts = getattr(batch, "request_counts", None)

        rows.append({
            "batch_id": batch.id,
            "scenario_name": row["scenario_name"],
            "model": row["model"],
            "processing_status": getattr(batch, "processing_status", None),
            "request_counts": str(request_counts),
            "created_at": str(getattr(batch, "created_at", None)),
            "ended_at": str(getattr(batch, "ended_at", None)),
            "expires_at": str(getattr(batch, "expires_at", None)),
            "plan_csv_path": row["plan_csv_path"],
        })

    return pd.DataFrame(rows)


def extract_text_from_anthropic_message(message) -> str:
    texts = []

    for block in getattr(message, "content", []) or []:
        if getattr(block, "type", None) == "text":
            texts.append(getattr(block, "text", ""))

    return "\n".join(t for t in texts if t).strip()


def anthropic_usage_to_dict(message) -> dict:
    usage = getattr(message, "usage", None)
    if usage is None:
        return {}

    try:
        return usage.model_dump()
    except Exception:
        try:
            return dict(usage)
        except Exception:
            return {}


def parse_anthropic_batch_results_to_standard_jsonl(
    batch_id: str,
    plan_csv_path: str | Path,
    scenario_name: str,
    config: ProviderModelConfig,
    output_dir: Path = AI_DATA_DIR,
) -> Path:
    plan_df = pd.read_csv(plan_csv_path)
    plan_by_custom_id = {
        row["anthropic_custom_id"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    standard_output_path = output_dir / f"{config.provider}__{config.model}__{scenario_name}.jsonl"
    raw_results_path = ANTHROPIC_BATCH_DIR / f"{batch_id}__results_raw.jsonl"

    n_success = 0
    n_error = 0

    result_stream = anthropic_client.messages.batches.results(batch_id)

    for entry in result_stream:
        try:
            entry_dict = entry.model_dump()
        except Exception:
            entry_dict = {"repr": repr(entry)}

        append_jsonl(raw_results_path, entry_dict)

        custom_id = getattr(entry, "custom_id", None)
        plan_row = plan_by_custom_id.get(custom_id, {})
        result = getattr(entry, "result", None)
        result_type = getattr(result, "type", None)

        if result_type == "succeeded":
            message = getattr(result, "message", None)
            text = extract_text_from_anthropic_message(message)
            usage = anthropic_usage_to_dict(message)

            record = {
                **plan_row,
                "created_at_utc": now_iso(),
                "status": "success" if text else "empty_text",
                "text": text if text else None,
                "provider_response_id": getattr(message, "id", None),
                "usage": usage,
                "raw_response": None,
                "error": None if text else "No text extracted from Anthropic message.",
                "batch_custom_id": custom_id,
                "batch_id": batch_id,
                "batch_output_file": str(raw_results_path),
            }

            n_success += int(bool(text))
            n_error += int(not bool(text))
        else:
            error_obj = getattr(result, "error", None)
            try:
                error_dump = error_obj.model_dump() if error_obj is not None else None
            except Exception:
                error_dump = repr(error_obj)

            record = {
                **plan_row,
                "created_at_utc": now_iso(),
                "status": "error",
                "text": None,
                "provider_response_id": None,
                "usage": None,
                "raw_response": None,
                "error": error_dump,
                "batch_custom_id": custom_id,
                "batch_id": batch_id,
                "batch_output_file": str(raw_results_path),
            }

            n_error += 1

        append_jsonl(standard_output_path, record)

    print(f"Raw Anthropic results: {raw_results_path}")
    print(f"Parsed standard output: {standard_output_path}")
    print(f"Success-ish text records: {n_success:,}")
    print(f"Errors/empty:             {n_error:,}")

    return standard_output_path

In [34]:
claude_cfg = MODEL_CONFIGS["claude_sonnet45"]

claude_main_batch = submit_anthropic_batch(
    "neutral_main_t1",
    claude_cfg,
    slogan_conditions_df,
)

claude_main_batch

Anthropic scenario: neutral_main_t1
Requests to submit: 50
Plan CSV: ai_data/slogan_batches/anthropic/anthropic__claude-sonnet-4-5__slogan__neutral_main_t1__20260429_170321__batch_plan.csv
Request JSON: ai_data/slogan_batches/anthropic/anthropic__claude-sonnet-4-5__slogan__neutral_main_t1__20260429_170321__batch_requests.json
Submitted Anthropic batch:
{
  "batch_id": "msgbatch_01W2vHxwpfm7rp8YHt6V9s8R",
  "scenario_name": "neutral_main_t1",
  "provider": "anthropic",
  "model": "claude-sonnet-4-5",
  "submitted_at_utc": "2026-04-29T21:03:21.997069+00:00",
  "n_requests_submitted": 50,
  "plan_csv_path": "ai_data/slogan_batches/anthropic/anthropic__claude-sonnet-4-5__slogan__neutral_main_t1__20260429_170321__batch_plan.csv",
  "request_json_path": "ai_data/slogan_batches/anthropic/anthropic__claude-sonnet-4-5__slogan__neutral_main_t1__20260429_170321__batch_requests.json",
  "processing_status": "in_progress"
}


{'batch_id': 'msgbatch_01W2vHxwpfm7rp8YHt6V9s8R',
 'scenario_name': 'neutral_main_t1',
 'provider': 'anthropic',
 'model': 'claude-sonnet-4-5',
 'submitted_at_utc': '2026-04-29T21:03:21.997069+00:00',
 'n_requests_submitted': 50,
 'plan_csv_path': 'ai_data/slogan_batches/anthropic/anthropic__claude-sonnet-4-5__slogan__neutral_main_t1__20260429_170321__batch_plan.csv',
 'request_json_path': 'ai_data/slogan_batches/anthropic/anthropic__claude-sonnet-4-5__slogan__neutral_main_t1__20260429_170321__batch_requests.json',
 'processing_status': 'in_progress'}

In [36]:
check_anthropic_batch(claude_main_batch["batch_id"])

MessageBatch(id='msgbatch_01W2vHxwpfm7rp8YHt6V9s8R', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 4, 29, 21, 3, 21, 888305, tzinfo=datetime.timezone.utc), ended_at=datetime.datetime(2026, 4, 29, 21, 5, 58, 97102, tzinfo=TzInfo(UTC)), expires_at=datetime.datetime(2026, 4, 30, 21, 3, 21, 888305, tzinfo=datetime.timezone.utc), processing_status='ended', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=50), results_url='https://api.anthropic.com/v1/messages/batches/msgbatch_01W2vHxwpfm7rp8YHt6V9s8R/results', type='message_batch')


MessageBatch(id='msgbatch_01W2vHxwpfm7rp8YHt6V9s8R', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 4, 29, 21, 3, 21, 888305, tzinfo=datetime.timezone.utc), ended_at=datetime.datetime(2026, 4, 29, 21, 5, 58, 97102, tzinfo=TzInfo(UTC)), expires_at=datetime.datetime(2026, 4, 30, 21, 3, 21, 888305, tzinfo=datetime.timezone.utc), processing_status='ended', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=50), results_url='https://api.anthropic.com/v1/messages/batches/msgbatch_01W2vHxwpfm7rp8YHt6V9s8R/results', type='message_batch')

In [37]:
batch = anthropic_client.messages.batches.retrieve(claude_main_batch["batch_id"])

if batch.processing_status != "ended":
    print("Claude main batch not ended yet.")
else:
    claude_main_standard_path = parse_anthropic_batch_results_to_standard_jsonl(
        batch_id=claude_main_batch["batch_id"],
        plan_csv_path=claude_main_batch["plan_csv_path"],
        scenario_name=claude_main_batch["scenario_name"],
        config=claude_cfg,
    )

    claude_main_df = load_generation_jsonl(claude_main_standard_path)
    print(claude_main_df.shape)
    print(claude_main_df["status"].value_counts(dropna=False))
    display(claude_main_df[["condition_id", "temperature", "run_idx", "status", "text"]].head(10))

Raw Anthropic results: ai_data/slogan_batches/anthropic/msgbatch_01W2vHxwpfm7rp8YHt6V9s8R__results_raw.jsonl
Parsed standard output: ai_data/slogan_generations/anthropic__claude-sonnet-4-5__neutral_main_t1.jsonl
Success-ish text records: 50
Errors/empty:             0
(50, 28)
status
success    50
Name: count, dtype: int64


,condition_id,temperature,run_idx,status,text
0,smartphone,1.0,7,success,Beyond screens. Into experiences.
1,smartphone,1.0,2,success,Beyond screens. Into possibilities.
2,smartphone,1.0,18,success,Beyond screens. Into life.
3,smartphone,1.0,21,success,Beyond screens. Into experiences.
4,smartphone,1.0,39,success,Beyond Smart. Purely Intuitive.
5,smartphone,1.0,13,success,Beyond Smart. Beyond Limits. Beyond.
6,smartphone,1.0,20,success,Beyond Smart. Brilliantly Human.
7,smartphone,1.0,33,success,Beyond Screens. Into Pure Experience.
8,smartphone,1.0,24,success,Beyond screens. Into experiences.
9,smartphone,1.0,49,success,Beyond screens. Into experiences.


In [38]:
claude_main_df = add_slogan_validity_diagnostics(claude_main_df)

claude_main_df.groupby("condition_id").agg(
    n=("text", "size"),
    n_success=("status", lambda x: (x == "success").sum()),
    mean_words=("output_word_count", "mean"),
    median_words=("output_word_count", "median"),
    validity_rate=("valid_one_slogan_heuristic", "mean"),
)

,n,n_success,mean_words,median_words,validity_rate
condition_id,,,,,
smartphone,50,50,4.7,4.5,1.0


In [39]:
claude_temp_batch = submit_anthropic_batch(
    "neutral_temperature_robustness",
    claude_cfg,
    slogan_conditions_df,
)

claude_personality_batch = submit_anthropic_batch(
    "personality_grid",
    claude_cfg,
    slogan_conditions_df,
)

claude_batches = pd.DataFrame([claude_main_batch, claude_temp_batch, claude_personality_batch])
claude_manifest_path = ANTHROPIC_BATCH_DIR / "anthropic__claude-sonnet-4-5__slogan_batch_manifest.csv"
claude_batches.to_csv(claude_manifest_path, index=False)

claude_batches

Anthropic scenario: neutral_temperature_robustness
Requests to submit: 20
Plan CSV: ai_data/slogan_batches/anthropic/anthropic__claude-sonnet-4-5__slogan__neutral_temperature_robustness__20260429_170705__batch_plan.csv
Request JSON: ai_data/slogan_batches/anthropic/anthropic__claude-sonnet-4-5__slogan__neutral_temperature_robustness__20260429_170705__batch_requests.json
Submitted Anthropic batch:
{
  "batch_id": "msgbatch_01EMEjn3SRmzAMCjTnYxtgfh",
  "scenario_name": "neutral_temperature_robustness",
  "provider": "anthropic",
  "model": "claude-sonnet-4-5",
  "submitted_at_utc": "2026-04-29T21:07:05.549826+00:00",
  "n_requests_submitted": 20,
  "plan_csv_path": "ai_data/slogan_batches/anthropic/anthropic__claude-sonnet-4-5__slogan__neutral_temperature_robustness__20260429_170705__batch_plan.csv",
  "request_json_path": "ai_data/slogan_batches/anthropic/anthropic__claude-sonnet-4-5__slogan__neutral_temperature_robustness__20260429_170705__batch_requests.json",
  "processing_status": "

,batch_id,scenario_name,provider,model,submitted_at_utc,n_requests_submitted,plan_csv_path,request_json_path,processing_status
0,msgbatch_01W2vHxwpfm7rp8YHt6V9s8R,neutral_main_t1,anthropic,claude-sonnet-4-5,2026-04-29T21:03:21.997069+00:00,50,ai_data/slogan_batches/anthropic/anthropic__cl...,ai_data/slogan_batches/anthropic/anthropic__cl...,in_progress
1,msgbatch_01EMEjn3SRmzAMCjTnYxtgfh,neutral_temperature_robustness,anthropic,claude-sonnet-4-5,2026-04-29T21:07:05.549826+00:00,20,ai_data/slogan_batches/anthropic/anthropic__cl...,ai_data/slogan_batches/anthropic/anthropic__cl...,in_progress
2,msgbatch_01Efzvw4No2JsH9EbjgMHQ8V,personality_grid,anthropic,claude-sonnet-4-5,2026-04-29T21:07:06.222017+00:00,960,ai_data/slogan_batches/anthropic/anthropic__cl...,ai_data/slogan_batches/anthropic/anthropic__cl...,in_progress


In [41]:
check_anthropic_batches(claude_batches)

,batch_id,scenario_name,model,processing_status,request_counts,created_at,ended_at,expires_at,plan_csv_path
0,msgbatch_01W2vHxwpfm7rp8YHt6V9s8R,neutral_main_t1,claude-sonnet-4-5,ended,"MessageBatchRequestCounts(canceled=0, errored=...",2026-04-29 21:03:21.888305+00:00,2026-04-29 21:05:58.097102+00:00,2026-04-30 21:03:21.888305+00:00,ai_data/slogan_batches/anthropic/anthropic__cl...
1,msgbatch_01EMEjn3SRmzAMCjTnYxtgfh,neutral_temperature_robustness,claude-sonnet-4-5,ended,"MessageBatchRequestCounts(canceled=0, errored=...",2026-04-29 21:07:05.416045+00:00,2026-04-29 21:09:47.571440+00:00,2026-04-30 21:07:05.416045+00:00,ai_data/slogan_batches/anthropic/anthropic__cl...
2,msgbatch_01Efzvw4No2JsH9EbjgMHQ8V,personality_grid,claude-sonnet-4-5,ended,"MessageBatchRequestCounts(canceled=0, errored=...",2026-04-29 21:07:06.095769+00:00,2026-04-29 21:10:47.707628+00:00,2026-04-30 21:07:06.095769+00:00,ai_data/slogan_batches/anthropic/anthropic__cl...


In [42]:
for batch_info in [claude_temp_batch, claude_personality_batch]:
    batch = anthropic_client.messages.batches.retrieve(batch_info["batch_id"])

    if batch.processing_status != "ended":
        print(f"Skipping {batch_info['scenario_name']}; status={batch.processing_status}")
        continue

    parse_anthropic_batch_results_to_standard_jsonl(
        batch_id=batch_info["batch_id"],
        plan_csv_path=batch_info["plan_csv_path"],
        scenario_name=batch_info["scenario_name"],
        config=claude_cfg,
    )

Raw Anthropic results: ai_data/slogan_batches/anthropic/msgbatch_01EMEjn3SRmzAMCjTnYxtgfh__results_raw.jsonl
Parsed standard output: ai_data/slogan_generations/anthropic__claude-sonnet-4-5__neutral_temperature_robustness.jsonl
Success-ish text records: 20
Errors/empty:             0
Raw Anthropic results: ai_data/slogan_batches/anthropic/msgbatch_01Efzvw4No2JsH9EbjgMHQ8V__results_raw.jsonl
Parsed standard output: ai_data/slogan_generations/anthropic__claude-sonnet-4-5__personality_grid.jsonl
Success-ish text records: 960
Errors/empty:             0


In [43]:
claude_all_df = load_all_generation_files(
    provider_name="anthropic",
    model_name="claude-sonnet-4-5",
    output_dir=AI_DATA_DIR,
)

claude_all_df = deduplicate_generation_records(claude_all_df)
claude_all_df = add_slogan_validity_diagnostics(claude_all_df)

print("Final Claude slogan shape:", claude_all_df.shape)
print(claude_all_df["scenario_name"].value_counts(dropna=False))
print(claude_all_df["status"].value_counts(dropna=False))

print(
    claude_all_df
    .query("status == 'success'")
    .groupby(["scenario_name", "temperature"], dropna=False)
    .size()
)

save_provider_outputs("anthropic", "claude-sonnet-4-5", claude_all_df)

Final Claude slogan shape: (1030, 36)
scenario_name
personality_grid                  960
neutral_main_t1                    50
neutral_temperature_robustness     20
Name: count, dtype: int64
status
success    1030
Name: count, dtype: int64
scenario_name                   temperature
neutral_main_t1                 1.0             50
neutral_temperature_robustness  0.3             10
                                0.7             10
personality_grid                0.3            320
                                0.7            320
                                1.0            320
dtype: int64
Saved:
ai_data/processed/anthropic__claude-sonnet-4-5__slogan_generations_all_records_with_errors.pkl
ai_data/processed/anthropic__claude-sonnet-4-5__slogan_generations_all_records_with_errors.csv
ai_data/processed/anthropic__claude-sonnet-4-5__slogan_generations_success_only.pkl
ai_data/processed/anthropic__claude-sonnet-4-5__slogan_generations_success_only.csv


{'audit_pkl': PosixPath('ai_data/processed/anthropic__claude-sonnet-4-5__slogan_generations_all_records_with_errors.pkl'),
 'audit_csv': PosixPath('ai_data/processed/anthropic__claude-sonnet-4-5__slogan_generations_all_records_with_errors.csv'),
 'success_pkl': PosixPath('ai_data/processed/anthropic__claude-sonnet-4-5__slogan_generations_success_only.pkl'),
 'success_csv': PosixPath('ai_data/processed/anthropic__claude-sonnet-4-5__slogan_generations_success_only.csv')}

# Gemini

In [44]:
def make_gemini_custom_id(request_key: str) -> str:
    return "r_" + request_key[:48]


def build_gemini_generation_plan(
    scenario_name: str,
    config: ProviderModelConfig,
    conditions_df: pd.DataFrame,
) -> pd.DataFrame:
    scenarios = build_scenarios_for_provider(config)
    scenario = scenarios[scenario_name]

    plan_df = build_slogan_generation_plan(
        scenario=scenario,
        conditions_df=conditions_df,
        provider_name=config.provider,
        model_name=config.model,
    ).copy()

    plan_df["gemini_custom_id"] = plan_df["request_key"].map(make_gemini_custom_id)

    if plan_df["gemini_custom_id"].duplicated().any():
        raise ValueError("Duplicate gemini_custom_id found. Increase hash length.")

    return plan_df


def make_gemini_batch_jsonl_file(
    scenario_name: str,
    config: ProviderModelConfig,
    conditions_df: pd.DataFrame,
    output_dir: Path = GEMINI_BATCH_DIR,
    exclude_existing_successes: bool = True,
    thinking_budget: int = 0,
) -> tuple[Path, Path, pd.DataFrame]:
    standard_output_path = AI_DATA_DIR / f"{config.provider}__{config.model}__{scenario_name}.jsonl"

    plan_df = build_gemini_generation_plan(
        scenario_name=scenario_name,
        config=config,
        conditions_df=conditions_df,
    )

    if exclude_existing_successes and standard_output_path.exists():
        done_keys = successful_request_keys(standard_output_path)
        plan_df = plan_df[~plan_df["request_key"].isin(done_keys)].copy()

    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"{config.provider}__{config.model}__{TASK_FAMILY}__{scenario_name}__{timestamp}"

    plan_csv_path = output_dir / f"{stem}__batch_plan.csv"
    batch_jsonl_path = output_dir / f"{stem}__batch_input.jsonl"

    plan_df.to_csv(plan_csv_path, index=False)

    with open(batch_jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            request_obj = {
                "key": row["gemini_custom_id"],
                "request": {
                    "contents": [
                        {
                            "role": "user",
                            "parts": [{"text": row["user_prompt"]}],
                        }
                    ],
                    "systemInstruction": {
                        "parts": [{"text": row["system_instructions"]}]
                    },
                    "generationConfig": {
                        "temperature": float(row["temperature"]),
                        "maxOutputTokens": int(row["max_output_tokens"]),
                        "responseModalities": ["TEXT"],
                        "thinkingConfig": {
                            "thinkingBudget": int(thinking_budget)
                        },
                    },
                },
            }

            f.write(json.dumps(request_obj, ensure_ascii=False) + "\n")

    print(f"Gemini scenario: {scenario_name}")
    print(f"Requests written: {len(plan_df):,}")
    print(f"Model: {config.model}")
    print(f"maxOutputTokens: {config.max_output_tokens}")
    print(f"thinkingBudget: {thinking_budget}")
    print(f"Plan CSV:     {plan_csv_path}")
    print(f"Batch JSONL:  {batch_jsonl_path}")

    return batch_jsonl_path, plan_csv_path, plan_df


def submit_gemini_batch_from_jsonl(
    scenario_name: str,
    config: ProviderModelConfig,
    conditions_df: pd.DataFrame,
    output_dir: Path = GEMINI_BATCH_DIR,
    exclude_existing_successes: bool = True,
) -> dict:
    batch_jsonl_path, plan_csv_path, plan_df = make_gemini_batch_jsonl_file(
        scenario_name=scenario_name,
        config=config,
        conditions_df=conditions_df,
        output_dir=output_dir,
        exclude_existing_successes=exclude_existing_successes,
        thinking_budget=0,
    )

    if len(plan_df) == 0:
        return {
            "batch_name": None,
            "scenario_name": scenario_name,
            "provider": config.provider,
            "model": config.model,
            "submitted_at_utc": now_iso(),
            "n_requests_submitted": 0,
            "plan_csv_path": str(plan_csv_path),
            "batch_jsonl_path": str(batch_jsonl_path),
            "uploaded_file_name": None,
        }

    uploaded_file = gemini_client.files.upload(
        file=str(batch_jsonl_path),
        config={"mime_type": "application/jsonl"},
    )

    print(f"Uploaded file: {uploaded_file.name}")

    batch_job = gemini_client.batches.create(
        model=config.model,
        src=uploaded_file.name,
        config={"display_name": batch_jsonl_path.stem},
    )

    batch_info = {
        "batch_name": batch_job.name,
        "scenario_name": scenario_name,
        "provider": config.provider,
        "model": config.model,
        "submitted_at_utc": now_iso(),
        "n_requests_submitted": len(plan_df),
        "plan_csv_path": str(plan_csv_path),
        "batch_jsonl_path": str(batch_jsonl_path),
        "uploaded_file_name": uploaded_file.name,
        "raw_state": str(getattr(batch_job, "state", None)),
    }

    manifest_path = output_dir / f"{batch_jsonl_path.stem}__batch_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(batch_info, f, indent=2)

    print("Submitted Gemini batch:")
    print(json.dumps(batch_info, indent=2))

    return batch_info


def check_gemini_batch(batch_name: str):
    batch = gemini_client.batches.get(name=batch_name)
    print(batch)
    return batch


def download_gemini_batch_output_file(batch_name: str, output_dir: Path = GEMINI_BATCH_DIR) -> Path:
    job = gemini_client.batches.get(name=batch_name)

    print("Batch state:", getattr(job, "state", None))
    print("Batch error:", getattr(job, "error", None))
    print("Batch dest:", getattr(job, "dest", None))

    dest = getattr(job, "dest", None)
    file_name = getattr(dest, "file_name", None)

    if not file_name:
        raise ValueError(f"No dest.file_name found on batch job: {batch_name}")

    safe_name = batch_name.replace("/", "_")
    output_path = output_dir / f"{safe_name}__output.jsonl"

    downloaded_bytes = gemini_client.files.download(file=file_name)

    if not isinstance(downloaded_bytes, (bytes, bytearray)):
        raise TypeError(f"Expected bytes from files.download, got {type(downloaded_bytes)}")

    output_path.write_bytes(downloaded_bytes)

    print(f"Downloaded Gemini batch output to: {output_path}")
    print(f"File size: {output_path.stat().st_size:,} bytes")

    return output_path


def extract_text_from_gemini_response_dict(response: dict) -> str:
    if not isinstance(response, dict):
        return ""

    if response.get("text"):
        return str(response["text"]).strip()

    texts = []

    for cand in response.get("candidates", []) or []:
        content = cand.get("content", {}) or {}
        for part in content.get("parts", []) or []:
            if isinstance(part, dict) and part.get("text"):
                texts.append(part["text"])

    return "\n".join(texts).strip()


def extract_gemini_usage_dict(response: dict) -> dict:
    if not isinstance(response, dict):
        return {}

    return (
        response.get("usageMetadata")
        or response.get("usage_metadata")
        or response.get("usage")
        or {}
    )


def parse_gemini_downloaded_batch_jsonl_to_standard_jsonl(
    downloaded_output_path: Path,
    plan_csv_path: str | Path,
    scenario_name: str,
    config: ProviderModelConfig,
    output_dir: Path = AI_DATA_DIR,
) -> Path:
    plan_df = pd.read_csv(plan_csv_path)
    plan_by_custom_id = {
        row["gemini_custom_id"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    raw_records = read_jsonl(downloaded_output_path)

    standard_output_path = output_dir / f"{config.provider}__{config.model}__{scenario_name}.jsonl"

    n_success = 0
    n_error = 0

    for rec in raw_records:
        custom_id = rec.get("key") or rec.get("custom_id") or rec.get("customId")
        plan_row = plan_by_custom_id.get(custom_id, {})

        response = rec.get("response") or rec.get("result") or {}
        error = rec.get("error")

        if isinstance(response, dict) and "body" in response:
            response = response.get("body") or {}

        text = extract_text_from_gemini_response_dict(response)
        usage = extract_gemini_usage_dict(response)

        if text:
            status = "success"
            n_success += 1
            error_out = None
        else:
            status = "error" if error else "empty_text"
            n_error += 1
            error_out = error or "No text extracted from Gemini response."

        record = {
            **plan_row,
            "created_at_utc": now_iso(),
            "status": status,
            "text": text if text else None,
            "provider_response_id": response.get("responseId") if isinstance(response, dict) else None,
            "usage": usage,
            "raw_response": None,
            "error": error_out,
            "batch_custom_id": custom_id,
            "batch_name": None,
            "batch_output_file": str(downloaded_output_path),
        }

        append_jsonl(standard_output_path, record)

    print(f"Parsed Gemini batch output to: {standard_output_path}")
    print(f"Success-ish text records: {n_success:,}")
    print(f"Errors/empty:             {n_error:,}")

    return standard_output_path

In [45]:
gemini_cfg = MODEL_CONFIGS["gemini_flash25"]

test_condition = slogan_conditions_df.iloc[0]

test_response = gemini_client.models.generate_content(
    model=gemini_cfg.model,
    contents=[
        types.Content(
            role="user",
            parts=[types.Part(text=build_slogan_user_prompt(test_condition["product"]))]
        )
    ],
    config=types.GenerateContentConfig(
        system_instruction=SLOGAN_NEUTRAL_SYSTEM_INSTRUCTIONS,
        temperature=1.3,
        max_output_tokens=40,
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)

print(test_response.text)

Your World, Enhanced.


In [46]:
gemini_main_batch = submit_gemini_batch_from_jsonl(
    scenario_name="neutral_main_t1",
    config=gemini_cfg,
    conditions_df=slogan_conditions_df,
)

gemini_main_batch

Gemini scenario: neutral_main_t1
Requests written: 50
Model: gemini-2.5-flash
maxOutputTokens: 40
thinkingBudget: 0
Plan CSV:     ai_data/slogan_batches/gemini/gemini__gemini-2.5-flash__slogan__neutral_main_t1__20260429_172728__batch_plan.csv
Batch JSONL:  ai_data/slogan_batches/gemini/gemini__gemini-2.5-flash__slogan__neutral_main_t1__20260429_172728__batch_input.jsonl
Uploaded file: files/ex1vwkm6o8u1
Submitted Gemini batch:
{
  "batch_name": "batches/68121rbyj0ea3nfhrrujc8ozw0agvhueugxy",
  "scenario_name": "neutral_main_t1",
  "provider": "gemini",
  "model": "gemini-2.5-flash",
  "submitted_at_utc": "2026-04-29T21:27:30.414817+00:00",
  "n_requests_submitted": 50,
  "plan_csv_path": "ai_data/slogan_batches/gemini/gemini__gemini-2.5-flash__slogan__neutral_main_t1__20260429_172728__batch_plan.csv",
  "batch_jsonl_path": "ai_data/slogan_batches/gemini/gemini__gemini-2.5-flash__slogan__neutral_main_t1__20260429_172728__batch_input.jsonl",
  "uploaded_file_name": "files/ex1vwkm6o8u1",


{'batch_name': 'batches/68121rbyj0ea3nfhrrujc8ozw0agvhueugxy',
 'scenario_name': 'neutral_main_t1',
 'provider': 'gemini',
 'model': 'gemini-2.5-flash',
 'submitted_at_utc': '2026-04-29T21:27:30.414817+00:00',
 'n_requests_submitted': 50,
 'plan_csv_path': 'ai_data/slogan_batches/gemini/gemini__gemini-2.5-flash__slogan__neutral_main_t1__20260429_172728__batch_plan.csv',
 'batch_jsonl_path': 'ai_data/slogan_batches/gemini/gemini__gemini-2.5-flash__slogan__neutral_main_t1__20260429_172728__batch_input.jsonl',
 'uploaded_file_name': 'files/ex1vwkm6o8u1',
 'raw_state': 'JobState.JOB_STATE_PENDING'}

In [48]:
check_gemini_batch(gemini_main_batch["batch_name"])

name='batches/68121rbyj0ea3nfhrrujc8ozw0agvhueugxy' display_name='gemini__gemini-2.5-flash__slogan__neutral_main_t1__20260429_172728__batch_input' state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'> error=None create_time=datetime.datetime(2026, 4, 29, 21, 27, 29, 899091, tzinfo=TzInfo(UTC)) start_time=None end_time=datetime.datetime(2026, 4, 29, 21, 27, 42, 698327, tzinfo=TzInfo(UTC)) update_time=datetime.datetime(2026, 4, 29, 21, 27, 42, 698327, tzinfo=TzInfo(UTC)) model='models/gemini-2.5-flash' src=None dest=BatchJobDestination(
  file_name='files/batch-68121rbyj0ea3nfhrrujc8ozw0agvhueugxy'
)


BatchJob(
  create_time=datetime.datetime(2026, 4, 29, 21, 27, 29, 899091, tzinfo=TzInfo(UTC)),
  dest=BatchJobDestination(
    file_name='files/batch-68121rbyj0ea3nfhrrujc8ozw0agvhueugxy'
  ),
  display_name='gemini__gemini-2.5-flash__slogan__neutral_main_t1__20260429_172728__batch_input',
  end_time=datetime.datetime(2026, 4, 29, 21, 27, 42, 698327, tzinfo=TzInfo(UTC)),
  model='models/gemini-2.5-flash',
  name='batches/68121rbyj0ea3nfhrrujc8ozw0agvhueugxy',
  state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'>,
  update_time=datetime.datetime(2026, 4, 29, 21, 27, 42, 698327, tzinfo=TzInfo(UTC))
)

In [49]:
gemini_main_output_path = download_gemini_batch_output_file(
    gemini_main_batch["batch_name"],
    output_dir=GEMINI_BATCH_DIR,
)

gemini_main_standard_path = parse_gemini_downloaded_batch_jsonl_to_standard_jsonl(
    downloaded_output_path=gemini_main_output_path,
    plan_csv_path=gemini_main_batch["plan_csv_path"],
    scenario_name=gemini_main_batch["scenario_name"],
    config=gemini_cfg,
)

gemini_main_df = load_generation_jsonl(gemini_main_standard_path)

print(gemini_main_df.shape)
print(gemini_main_df["status"].value_counts(dropna=False))
display(gemini_main_df[["condition_id", "temperature", "run_idx", "status", "text"]].head(10))

Batch state: JobState.JOB_STATE_SUCCEEDED
Batch error: None
Batch dest: format=None gcs_uri=None bigquery_uri=None file_name='files/batch-68121rbyj0ea3nfhrrujc8ozw0agvhueugxy' inlined_responses=None inlined_embed_content_responses=None
Downloaded Gemini batch output to: ai_data/slogan_batches/gemini/batches_68121rbyj0ea3nfhrrujc8ozw0agvhueugxy__output.jsonl
File size: 20,827 bytes
Parsed Gemini batch output to: ai_data/slogan_generations/gemini__gemini-2.5-flash__neutral_main_t1.jsonl
Success-ish text records: 50
Errors/empty:             0
(50, 28)
status
success    50
Name: count, dtype: int64


,condition_id,temperature,run_idx,status,text
0,smartphone,1.0,0,success,Future in your palm.
1,smartphone,1.0,1,success,"Your World, Amplified."
2,smartphone,1.0,2,success,Future in your hand.
3,smartphone,1.0,3,success,"Your world, in your pocket."
4,smartphone,1.0,4,success,"Your world, reimagined."
5,smartphone,1.0,5,success,"Your world, simplified."
6,smartphone,1.0,6,success,Future in your pocket.
7,smartphone,1.0,7,success,"Beyond Smart, It's Intuitive."
8,smartphone,1.0,8,success,Future in your hand.
9,smartphone,1.0,9,success,Future in your hand.


In [50]:
gemini_main_df = add_slogan_validity_diagnostics(gemini_main_df)

gemini_main_df.groupby("condition_id").agg(
    n=("text", "size"),
    n_success=("status", lambda x: (x == "success").sum()),
    mean_words=("output_word_count", "mean"),
    median_words=("output_word_count", "median"),
    validity_rate=("valid_one_slogan_heuristic", "mean"),
)

,n,n_success,mean_words,median_words,validity_rate
condition_id,,,,,
smartphone,50,50,3.84,4.0,1.0


In [51]:
gemini_temp_batch = submit_gemini_batch_from_jsonl(
    scenario_name="neutral_temperature_robustness",
    config=gemini_cfg,
    conditions_df=slogan_conditions_df,
)

gemini_personality_batch = submit_gemini_batch_from_jsonl(
    scenario_name="personality_grid",
    config=gemini_cfg,
    conditions_df=slogan_conditions_df,
)

gemini_batches = pd.DataFrame([gemini_main_batch, gemini_temp_batch, gemini_personality_batch])
gemini_manifest_path = GEMINI_BATCH_DIR / "gemini__gemini-2.5-flash__slogan_batch_manifest.csv"
gemini_batches.to_csv(gemini_manifest_path, index=False)

gemini_batches

Gemini scenario: neutral_temperature_robustness
Requests written: 20
Model: gemini-2.5-flash
maxOutputTokens: 40
thinkingBudget: 0
Plan CSV:     ai_data/slogan_batches/gemini/gemini__gemini-2.5-flash__slogan__neutral_temperature_robustness__20260429_173845__batch_plan.csv
Batch JSONL:  ai_data/slogan_batches/gemini/gemini__gemini-2.5-flash__slogan__neutral_temperature_robustness__20260429_173845__batch_input.jsonl
Uploaded file: files/cz5hug3qzcle
Submitted Gemini batch:
{
  "batch_name": "batches/p6j92x7jg8scyxztqwfz4gpx6d8dgcv91u26",
  "scenario_name": "neutral_temperature_robustness",
  "provider": "gemini",
  "model": "gemini-2.5-flash",
  "submitted_at_utc": "2026-04-29T21:38:49.030755+00:00",
  "n_requests_submitted": 20,
  "plan_csv_path": "ai_data/slogan_batches/gemini/gemini__gemini-2.5-flash__slogan__neutral_temperature_robustness__20260429_173845__batch_plan.csv",
  "batch_jsonl_path": "ai_data/slogan_batches/gemini/gemini__gemini-2.5-flash__slogan__neutral_temperature_robus

,batch_name,scenario_name,provider,model,submitted_at_utc,n_requests_submitted,plan_csv_path,batch_jsonl_path,uploaded_file_name,raw_state
0,batches/68121rbyj0ea3nfhrrujc8ozw0agvhueugxy,neutral_main_t1,gemini,gemini-2.5-flash,2026-04-29T21:27:30.414817+00:00,50,ai_data/slogan_batches/gemini/gemini__gemini-2...,ai_data/slogan_batches/gemini/gemini__gemini-2...,files/ex1vwkm6o8u1,JobState.JOB_STATE_PENDING
1,batches/p6j92x7jg8scyxztqwfz4gpx6d8dgcv91u26,neutral_temperature_robustness,gemini,gemini-2.5-flash,2026-04-29T21:38:49.030755+00:00,20,ai_data/slogan_batches/gemini/gemini__gemini-2...,ai_data/slogan_batches/gemini/gemini__gemini-2...,files/cz5hug3qzcle,JobState.JOB_STATE_PENDING
2,batches/1ocaz7cpk8opy7p1chutki5grdi2e8za9k79,personality_grid,gemini,gemini-2.5-flash,2026-04-29T21:38:51.283636+00:00,960,ai_data/slogan_batches/gemini/gemini__gemini-2...,ai_data/slogan_batches/gemini/gemini__gemini-2...,files/g7zvjq6imrrx,JobState.JOB_STATE_PENDING


In [53]:
for _, row in gemini_batches.dropna(subset=["batch_name"]).iterrows():
    print("=" * 100)
    print(row["scenario_name"])
    check_gemini_batch(row["batch_name"])

neutral_main_t1
name='batches/68121rbyj0ea3nfhrrujc8ozw0agvhueugxy' display_name='gemini__gemini-2.5-flash__slogan__neutral_main_t1__20260429_172728__batch_input' state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'> error=None create_time=datetime.datetime(2026, 4, 29, 21, 27, 29, 899091, tzinfo=TzInfo(UTC)) start_time=None end_time=datetime.datetime(2026, 4, 29, 21, 27, 42, 698327, tzinfo=TzInfo(UTC)) update_time=datetime.datetime(2026, 4, 29, 21, 27, 42, 698327, tzinfo=TzInfo(UTC)) model='models/gemini-2.5-flash' src=None dest=BatchJobDestination(
  file_name='files/batch-68121rbyj0ea3nfhrrujc8ozw0agvhueugxy'
)
neutral_temperature_robustness
name='batches/p6j92x7jg8scyxztqwfz4gpx6d8dgcv91u26' display_name='gemini__gemini-2.5-flash__slogan__neutral_temperature_robustness__20260429_173845__batch_input' state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'> error=None create_time=datetime.datetime(2026, 4, 29, 21, 38, 48, 379598, tzinfo=TzInfo(UTC)) start_time=None end_time=

In [54]:
for batch_info in [gemini_temp_batch, gemini_personality_batch]:
    job = gemini_client.batches.get(name=batch_info["batch_name"])

    if str(getattr(job, "state", "")) != "JobState.JOB_STATE_SUCCEEDED":
        print(f"Skipping {batch_info['scenario_name']}; state={getattr(job, 'state', None)}")
        continue

    output_path = download_gemini_batch_output_file(
        batch_info["batch_name"],
        output_dir=GEMINI_BATCH_DIR,
    )

    parse_gemini_downloaded_batch_jsonl_to_standard_jsonl(
        downloaded_output_path=output_path,
        plan_csv_path=batch_info["plan_csv_path"],
        scenario_name=batch_info["scenario_name"],
        config=gemini_cfg,
    )

Batch state: JobState.JOB_STATE_SUCCEEDED
Batch error: None
Batch dest: format=None gcs_uri=None bigquery_uri=None file_name='files/batch-p6j92x7jg8scyxztqwfz4gpx6d8dgcv91u26' inlined_responses=None inlined_embed_content_responses=None
Downloaded Gemini batch output to: ai_data/slogan_batches/gemini/batches_p6j92x7jg8scyxztqwfz4gpx6d8dgcv91u26__output.jsonl
File size: 8,319 bytes
Parsed Gemini batch output to: ai_data/slogan_generations/gemini__gemini-2.5-flash__neutral_temperature_robustness.jsonl
Success-ish text records: 20
Errors/empty:             0
Batch state: JobState.JOB_STATE_SUCCEEDED
Batch error: None
Batch dest: format=None gcs_uri=None bigquery_uri=None file_name='files/batch-1ocaz7cpk8opy7p1chutki5grdi2e8za9k79' inlined_responses=None inlined_embed_content_responses=None
Downloaded Gemini batch output to: ai_data/slogan_batches/gemini/batches_1ocaz7cpk8opy7p1chutki5grdi2e8za9k79__output.jsonl
File size: 406,672 bytes
Parsed Gemini batch output to: ai_data/slogan_generati

In [55]:
gemini_all_df = load_all_generation_files("gemini", "gemini-2.5-flash", AI_DATA_DIR)
gemini_all_df = deduplicate_generation_records(gemini_all_df)
gemini_all_df = add_slogan_validity_diagnostics(gemini_all_df)

print(gemini_all_df["scenario_name"].value_counts(dropna=False))
print(gemini_all_df["status"].value_counts(dropna=False))

save_provider_outputs("gemini", "gemini-2.5-flash", gemini_all_df)

scenario_name
personality_grid                  960
neutral_main_t1                    50
neutral_temperature_robustness     20
Name: count, dtype: int64
status
success    1030
Name: count, dtype: int64
Saved:
ai_data/processed/gemini__gemini-2.5-flash__slogan_generations_all_records_with_errors.pkl
ai_data/processed/gemini__gemini-2.5-flash__slogan_generations_all_records_with_errors.csv
ai_data/processed/gemini__gemini-2.5-flash__slogan_generations_success_only.pkl
ai_data/processed/gemini__gemini-2.5-flash__slogan_generations_success_only.csv


{'audit_pkl': PosixPath('ai_data/processed/gemini__gemini-2.5-flash__slogan_generations_all_records_with_errors.pkl'),
 'audit_csv': PosixPath('ai_data/processed/gemini__gemini-2.5-flash__slogan_generations_all_records_with_errors.csv'),
 'success_pkl': PosixPath('ai_data/processed/gemini__gemini-2.5-flash__slogan_generations_success_only.pkl'),
 'success_csv': PosixPath('ai_data/processed/gemini__gemini-2.5-flash__slogan_generations_success_only.csv')}

In [56]:
PROVIDER_MODELS = [
    ("openai", "gpt-5.4"),
    ("anthropic", "claude-sonnet-4-5"),
    ("gemini", "gemini-2.5-flash"),
]

all_model_dfs = []

for provider, model in PROVIDER_MODELS:
    df = load_all_generation_files(provider, model, AI_DATA_DIR)
    if df.empty:
        print(f"No files found for {provider} / {model}")
        continue

    df = deduplicate_generation_records(df)
    df = add_slogan_validity_diagnostics(df)
    all_model_dfs.append(df)

combined_slogan_df = pd.concat(all_model_dfs, ignore_index=True)

combined_slogan_df["analysis_scenario_name"] = combined_slogan_df["scenario_name"]

summary = (
    combined_slogan_df
    .groupby(["provider", "model", "analysis_scenario_name", "temperature", "status"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["provider", "model", "analysis_scenario_name", "temperature", "status"])
)

summary

,provider,model,analysis_scenario_name,temperature,status,n
0,anthropic,claude-sonnet-4-5,neutral_main_t1,1.0,success,50
1,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,0.3,success,10
2,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,0.7,success,10
3,anthropic,claude-sonnet-4-5,personality_grid,0.3,success,320
4,anthropic,claude-sonnet-4-5,personality_grid,0.7,success,320
5,anthropic,claude-sonnet-4-5,personality_grid,1.0,success,320
6,gemini,gemini-2.5-flash,neutral_main_t1,1.0,success,50
7,gemini,gemini-2.5-flash,neutral_temperature_robustness,0.7,success,10
8,gemini,gemini-2.5-flash,neutral_temperature_robustness,1.3,success,10
9,gemini,gemini-2.5-flash,personality_grid,0.7,success,320


In [57]:
success_counts = (
    combined_slogan_df
    .query("status == 'success'")
    .groupby(["provider", "model", "analysis_scenario_name", "temperature"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["provider", "model", "analysis_scenario_name", "temperature"])
)

success_counts

,provider,model,analysis_scenario_name,temperature,n
0,anthropic,claude-sonnet-4-5,neutral_main_t1,1.0,50
1,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,0.3,10
2,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,0.7,10
3,anthropic,claude-sonnet-4-5,personality_grid,0.3,320
4,anthropic,claude-sonnet-4-5,personality_grid,0.7,320
5,anthropic,claude-sonnet-4-5,personality_grid,1.0,320
6,gemini,gemini-2.5-flash,neutral_main_t1,1.0,50
7,gemini,gemini-2.5-flash,neutral_temperature_robustness,0.7,10
8,gemini,gemini-2.5-flash,neutral_temperature_robustness,1.3,10
9,gemini,gemini-2.5-flash,personality_grid,0.7,320


In [58]:
combined_slogan_df = deduplicate_generation_records(combined_slogan_df)
combined_slogan_df = add_slogan_validity_diagnostics(combined_slogan_df)

combined_success_df = combined_slogan_df.query("status == 'success'").copy()

combined_audit_pkl = CLEAN_AI_DIR / "all_models__slogan_generations_all_records_with_errors.pkl"
combined_audit_csv = CLEAN_AI_DIR / "all_models__slogan_generations_all_records_with_errors.csv"
combined_success_pkl = CLEAN_AI_DIR / "all_models__slogan_generations_success_only.pkl"
combined_success_csv = CLEAN_AI_DIR / "all_models__slogan_generations_success_only.csv"

combined_slogan_df.to_pickle(combined_audit_pkl)
combined_slogan_df.to_csv(combined_audit_csv, index=False)

combined_success_df.to_pickle(combined_success_pkl)
combined_success_df.to_csv(combined_success_csv, index=False)

print("Saved combined slogan files:")
print(combined_audit_pkl)
print(combined_audit_csv)
print(combined_success_pkl)
print(combined_success_csv)

Saved combined slogan files:
ai_data/processed/all_models__slogan_generations_all_records_with_errors.pkl
ai_data/processed/all_models__slogan_generations_all_records_with_errors.csv
ai_data/processed/all_models__slogan_generations_success_only.pkl
ai_data/processed/all_models__slogan_generations_success_only.csv


In [59]:
def show_slogan_examples(
    df: pd.DataFrame,
    provider: Optional[str] = None,
    model: Optional[str] = None,
    scenario_name: Optional[str] = None,
    temperature: Optional[float] = None,
    n: int = 20,
    random_state: int = 42,
):
    use = df.copy()

    if provider is not None:
        use = use[use["provider"] == provider]
    if model is not None:
        use = use[use["model"] == model]
    if scenario_name is not None:
        use = use[use["scenario_name"] == scenario_name]
    if temperature is not None:
        use = use[use["temperature"] == temperature]

    use = use[use["status"] == "success"].copy()

    if len(use) == 0:
        print("No matching successful generations.")
        return

    sample = use.sample(n=min(n, len(use)), random_state=random_state)

    for i, (_, row) in enumerate(sample.iterrows(), start=1):
        print("=" * 80)
        print(f"Example {i}")
        print(f"Provider/model: {row['provider']} / {row['model']}")
        print(f"Scenario: {row['scenario_name']}")
        print(f"Temperature: {row['temperature']}")
        print(f"Persona: {row.get('persona_id')}")
        print(f"Words: {row.get('output_word_count')}")
        print("-" * 80)
        print(row["text"])


show_slogan_examples(
    combined_success_df,
    scenario_name="neutral_main_t1",
    n=20,
)

Example 1
Provider/model: openai / gpt-5.4
Scenario: neutral_main_t1
Temperature: 1.0
Persona: nan
Words: 5
--------------------------------------------------------------------------------
Tomorrow, perfectly in your palm.
Example 2
Provider/model: gemini / gemini-2.5-flash
Scenario: neutral_main_t1
Temperature: 1.0
Persona: nan
Words: 4
--------------------------------------------------------------------------------
Future in your hand.
Example 3
Provider/model: gemini / gemini-2.5-flash
Scenario: neutral_main_t1
Temperature: 1.0
Persona: nan
Words: 4
--------------------------------------------------------------------------------
See Beyond The Edge.
Example 4
Provider/model: anthropic / claude-sonnet-4-5
Scenario: neutral_main_t1
Temperature: 1.0
Persona: nan
Words: 5
--------------------------------------------------------------------------------
Beyond Smart. Beyond Limits. Beyond.
Example 5
Provider/model: anthropic / claude-sonnet-4-5
Scenario: neutral_main_t1
Temperature: 1.0
P